# ĐỒ ÁN TỐT NGHIỆP: TỐI ƯU HÓA KÊNH TRUYỀN LORA VỚI TINYML (CHANNEL HOPPING)

### Kiến trúc phần cứng:
- **3 End-Nodes:** ESP32 DevKit V1 + Module LoRa Ra-02 (SX1278 433 MHz) + OLED SSD1306
- **1 Multi-Channel Gateway:** Raspberry Pi CM4 + 3 Module Ra-02 (lắng nghe 3 kênh 433.175, 433.375, 433.575 MHz)

### Pipeline xử lý dữ liệu:
1. Thu thập log từ Gateway → Parse thành CSV chuẩn
2. Phát hiện gói mất → Gán nhãn RSSI/SNR = -1000
3. Tạo dataset chuỗi thời gian (Sliding Window ts=10)
4. Huấn luyện mô hình FCNN → Xuất TFLite → model.h cho ESP32
5. Đánh giá so sánh: Fixed vs Random vs TinyML Hopping
6. Benchmark các kiến trúc mạng nơ-ron
7. Đánh giá chuyên sâu mô hình tốt nhất


## 1. Khởi tạo môi trường & Cấu hình đường dẫn


In [ ]:
import os
import re
import json
import shutil
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import regularizers, layers, models

SEED = 1337
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
LOGS_DIR = os.path.join(DATA_DIR, "logs")
PARSED_DIR = os.path.join(DATA_DIR, "parsed")
IMPUTED_DIR = os.path.join(DATA_DIR, "imputed")
READY_DIR = os.path.join(DATA_DIR, "ready")
FORMATTED_DIR = os.path.join(DATA_DIR, "formattedCSV")
MLDATAREADY_DIR = os.path.join(DATA_DIR, "MLdataready")
MODELS_DIR = os.path.join(BASE_DIR, "models")
FIGURES_DIR = os.path.join(BASE_DIR, "figures")

for d in [LOGS_DIR, PARSED_DIR, IMPUTED_DIR, READY_DIR, MODELS_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"[OK] TensorFlow: {tf.__version__}")
print(f"[OK] Thư mục: {BASE_DIR}")


## 2. Bước 1: Trích xuất và chuẩn hóa Log Gateway (`01_data_parser`)
Regex hỗ trợ 2 dạng log:
- **Pipe-separated** (từ main_gw.cpp): `12:00:00.123 | A | 0 | 30 B | -75 | 9.0 | 433.175`
- **Text log**: `11:45:51.918 -> A0* size_of_message 30 RSSI -88 SNR 10 Frequency 433.175`


In [ ]:
PATTERN_PIPE = re.compile(
    r"(?P<time_slot>\d{2}:\d{2}:\d{2}\.\d{3})\s*\|\s*"
    r"(?P<source>[A-Z])\s*\|\s*"
    r"(?P<packet_ID>\d+)\s*\|\s*"
    r"(?P<size_of_message>\d+)\s*(?:B)?\s*\|\s*"
    r"(?P<RSSI>-?\d+)\s*\|\s*"
    r"(?P<SNR>-?\d+(?:\.\d+)?)\s*\|\s*"
    r"(?P<frequency>\d{3}\.\d{3})"
)

PATTERN_TEXT = re.compile(
    r"(?P<time_slot>\d{2}:\d{2}:\d{2}\.\d{3})\s*->\s*"
    r"(?P<source>[A-Z])(?P<packet_ID>\d+)(?:\*.*?)\s*.*?"
    r"size_of_message\s+(?P<size_of_message>\d+)\s*"
    r"RSSI\s+(?P<RSSI>-?\d+)\s*"
    r"SNR\s+(?P<SNR>-?\d+(?:\.\d+)?)\s*"
    r"Frequency\s+(?P<frequency>\d+\.\d+)",
    re.IGNORECASE
)

REQUIRED_COLS = ['time_slot', 'source', 'packet_ID', 'size_of_message', 'RSSI', 'SNR', 'frequency']

parsed_dfs = []
log_files = [f for f in os.listdir(LOGS_DIR) if f.endswith(('.csv', '.log', '.txt'))]
print(f"[INFO] Tìm thấy {len(log_files)} file log trong logs/")

for fname in log_files:
    fpath = os.path.join(LOGS_DIR, fname)

    # Thử đọc CSV trực tiếp
    try:
        df = pd.read_csv(fpath)
        df.columns = [c.strip() for c in df.columns]
        if all(c in df.columns for c in ['source', 'packet_ID', 'RSSI', 'SNR', 'frequency']):
            if 'time_slot' not in df.columns:
                df['time_slot'] = '00:00:00.000'
            df = df[REQUIRED_COLS]
            parsed_dfs.append(df)
            continue
    except Exception:
        pass

    # Parse text line-by-line
    data = []
    with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line_str = line.strip()
            if line_str.startswith("---") or line_str.startswith("===") or "Time" in line_str:
                continue
            match = PATTERN_PIPE.search(line_str)
            if not match:
                match = PATTERN_TEXT.search(line_str)
            if match:
                d = match.groupdict()
                d["packet_ID"] = int(d["packet_ID"])
                d["size_of_message"] = int(d["size_of_message"])
                d["RSSI"] = int(d["RSSI"])
                d["SNR"] = float(d["SNR"])
                d["frequency"] = round(float(d["frequency"]), 3)
                data.append(d)

    if data:
        parsed_dfs.append(pd.DataFrame(data)[REQUIRED_COLS])
        print(f"  [OK] {fname}: {len(data)} dòng")

if parsed_dfs:
    raw_combined = pd.concat(parsed_dfs, ignore_index=True)
    parsed_out = os.path.join(PARSED_DIR, "raw_parsed.csv")
    raw_combined.to_csv(parsed_out, index=False)
    print(f"\n[SUCCESS] Tổng hợp {len(raw_combined)} dòng -> {parsed_out}")
    raw_combined.head(8)


## 3. Bước 2: Phát hiện gói mất & Gán nhãn (`02_add_missed_packets`)
Gói tin bị mất (collision / suy hao) được gán: $RSSI = -1000, SNR = -1000$


In [ ]:
def impute_missed_packets(input_csv, output_csv):
    df = pd.read_csv(input_csv)
    df['packet_ID'] = pd.to_numeric(df['packet_ID'], errors='coerce').astype(int)

    new_rows = []
    for (source, size), sub in df.groupby(['source', 'size_of_message']):
        if sub.empty:
            continue
        freq = sub.iloc[0]['frequency']
        min_id, max_id = int(sub['packet_ID'].min()), int(sub['packet_ID'].max())
        missing = set(range(min_id, max_id + 1)) - set(sub['packet_ID'])

        for m_id in missing:
            match = df[(df['packet_ID'] == m_id) & (df['source'] != source)]
            t_slot = match.iloc[0]['time_slot'] if not match.empty else "00:00:00.000"
            f_val = match.iloc[0]['frequency'] if not match.empty else freq

            new_rows.append({
                'time_slot': t_slot, 'source': source, 'packet_ID': m_id,
                'size_of_message': size, 'RSSI': -1000, 'SNR': -1000, 'frequency': f_val
            })

    if new_rows:
        df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)

    df = df.sort_values(by=['source', 'size_of_message', 'packet_ID']).reset_index(drop=True)
    df.to_csv(output_csv, index=False)
    print(f"[OK] Bổ sung {len(new_rows)} gói mất -> {output_csv}")
    return df

imputed_out = os.path.join(IMPUTED_DIR, "imputed_packets.csv")
imputed_df = impute_missed_packets(os.path.join(PARSED_DIR, "raw_parsed.csv"), imputed_out)

total = len(imputed_df)
lost = len(imputed_df[imputed_df['RSSI'] == -1000])
print(f"Tổng: {total} | Nhận: {total - lost} | Mất: {lost} ({lost/total*100:.1f}%)")


## 4. Bước 3: Tạo dataset chuỗi thời gian (`03_generate_dataset`)
Mỗi mẫu: cửa sổ trượt $t_s = 10$ bước, mỗi bước 8 đặc trưng → 80 chiều input.
Nhãn: kênh có RSSI cao nhất tại bước $t_s + 1$.


In [ ]:
CHANNELS = [433.175, 433.375, 433.575]
WINDOW_SIZE = 10

def assign_ch_flags(row):
    freq = row['frequency']
    is_loss = (row['RSSI'] == -1000)
    ch_idx = int(np.argmin([abs(freq - c) for c in CHANNELS]))
    flags = [1, 1, 1]
    flags[ch_idx] = 1 if is_loss else 0
    return pd.Series(flags, index=['CH1', 'CH2', 'CH3'])

def generate_features_and_labels(df):
    flags_df = df.apply(assign_ch_flags, axis=1)
    df = pd.concat([df, flags_df], axis=1)

    best_lookup = {}
    for (src, size, pid), grp in df.groupby(['source', 'size_of_message', 'packet_ID']):
        valid = grp[grp['RSSI'] > -900]
        if not valid.empty:
            best_ch = int(np.argmin([abs(valid.loc[valid['RSSI'].idxmax(), 'frequency'] - c) for c in CHANNELS]))
        else:
            best_ch = int(pid % len(CHANNELS))
        best_lookup[(src, size, pid)] = best_ch

    samples = []
    for (source, size), group in df.groupby(['source', 'size_of_message']):
        group = group.sort_values('packet_ID').reset_index(drop=True)
        if len(group) <= WINDOW_SIZE:
            continue
        for i in range(len(group) - WINDOW_SIZE):
            window = group.iloc[i:i + WINDOW_SIZE]
            target = group.iloc[i + WINDOW_SIZE]
            features = []
            for _, r in window.iterrows():
                features.extend([float(r['CH1']), float(r['CH2']), float(r['CH3']),
                                 float(r['SNR']), float(r['RSSI']),
                                 float(r['size_of_message']), float(r['packet_ID']),
                                 float(r['frequency'])])
            pid = int(target['packet_ID'])
            label = best_lookup.get((source, size, pid), int(pid % len(CHANNELS)))
            d = {f"feat_{j}": v for j, v in enumerate(features)}
            d['target_channel'] = label
            d['source'] = source
            d['size_of_message'] = size
            samples.append(d)
    return pd.DataFrame(samples)

ready_df = generate_features_and_labels(imputed_df)
ready_out = os.path.join(READY_DIR, "dataset_ready.csv")
ready_df.to_csv(ready_out, index=False)
print(f"[SUCCESS] Dataset: {len(ready_df)} mẫu ({WINDOW_SIZE * 8} đặc trưng/mẫu)")
ready_df.head(5)


## 5. Bước 4: Huấn luyện FCNN & Xuất TinyML (`04_train_model`)
- Dense(10, relu) → Dense(10, relu, L1) → Dense(3, softmax)
- Adam(lr=0.005), Loss: categorical_crossentropy
- 60% Train, 20% Val, 20% Test


In [ ]:
feat_cols = [c for c in ready_df.columns if c.startswith("feat_")]
X = ready_df[feat_cols].values
y = ready_df['target_channel'].values
y_onehot = tf.keras.utils.to_categorical(y, num_classes=3)

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_temp, y_train, y_temp = train_test_split(X, y_onehot, test_size=0.4, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

model = tf.keras.Sequential([
    tf.keras.layers.Dense(10, activation='relu', input_shape=(X.shape[1],), name="dense_1"),
    tf.keras.layers.Dense(10, activation='relu', kernel_regularizer=regularizers.L1(1e-4), name="dense_2"),
    tf.keras.layers.Dense(3, activation='softmax', name="output")
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True, verbose=1)
history = model.fit(X_train, y_train, epochs=200, batch_size=16,
                    validation_data=(X_val, y_val), callbacks=[early_stop], verbose=1)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\n{'='*50}")
print(f" TEST ACCURACY: {test_acc * 100:.2f}% | LOSS: {test_loss:.4f}")
print(f"{'='*50}")


### Trực quan hóa đường cong học tập


In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy', color='#2ca02c', linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validation Accuracy', color='#1f77b4', linewidth=2)
plt.title('Accuracy Trends', fontsize=12)
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss', color='#d62728', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation Loss', color='#ff7f0e', linewidth=2)
plt.title('Loss Trends', fontsize=12)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.savefig(os.path.join(FIGURES_DIR, "training_trends.png"), dpi=300, bbox_inches='tight')
plt.show()


## 6. Chuyển đổi TFLite & Xuất `model.h` cho ESP32


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = os.path.join(MODELS_DIR, "channel_model.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)
print(f"[OK] TFLite: {tflite_path} ({len(tflite_model)} bytes ~ {len(tflite_model)/1024:.2f} KB)")

header_path = os.path.join(MODELS_DIR, "model.h")
with open(header_path, "w", encoding="utf-8") as f:
    f.write("// TinyML LoRa Channel Hopping Model\n")
    f.write(f"// Size: {len(tflite_model)} bytes\n\n")
    f.write("const unsigned char model[] = {\n  ")
    for i, b in enumerate(tflite_model):
        f.write(f"0x{b:02x}, ")
        if (i + 1) % 12 == 0:
            f.write("\n  ")
    f.write("\n};\n")
    f.write(f"const unsigned int model_len = {len(tflite_model)};\n")
print(f"[OK] C Header: {header_path}")

mcu_src = os.path.join(BASE_DIR, "../MCUCode/LoRaEN_TinyMLHopping/src/model.h")
if os.path.exists(os.path.dirname(mcu_src)):
    shutil.copyfile(header_path, mcu_src)
    print(f"[SYNC] -> {mcu_src}")


## 7. Bước 5: Đánh giá so sánh hiệu năng (`05_evaluate_comparison`)
So sánh 3 cơ chế: **Fixed Channel** / **Random Hopping** / **TinyML Hopping**

Dữ liệu được tính toán 100% từ các file CSV thực nghiệm (imputed hoặc formattedCSV).


In [ ]:
# Nạp dữ liệu thực nghiệm
eval_dfs = []
for src_dir in [IMPUTED_DIR, FORMATTED_DIR]:
    if os.path.exists(src_dir):
        for f in os.listdir(src_dir):
            if f.endswith(".csv"):
                eval_dfs.append(pd.read_csv(os.path.join(src_dir, f)))
    if eval_dfs:
        break

if not eval_dfs:
    print("[ERROR] Không tìm thấy dữ liệu thực nghiệm!")
else:
    eval_combined = pd.concat(eval_dfs, ignore_index=True)
    print(f"[DATA] {len(eval_combined)} dòng dữ liệu thực nghiệm")

    random.seed(SEED)
    payload_sizes = sorted(eval_combined['size_of_message'].unique())
    results = []

    for size in payload_sizes:
        sub = eval_combined[eval_combined['size_of_message'] == size]
        if sub.empty:
            continue

        # Fixed Channel
        pdr_fixed_list = []
        for ch in sub['frequency'].unique():
            ch_sub = sub[sub['frequency'] == ch]
            recv = len(ch_sub[ch_sub['RSSI'] > -900])
            if len(ch_sub) > 0:
                pdr_fixed_list.append(recv / len(ch_sub))
        pdr_fixed = float(np.mean(pdr_fixed_list)) if pdr_fixed_list else 0.0

        # Random & TinyML
        r_rssi, r_snr, r_total, r_ok = [], [], 0, 0
        m_rssi, m_snr, m_total, m_ok = [], [], 0, 0

        for pid in sub['packet_ID'].unique():
            slot = sub[sub['packet_ID'] == pid]
            if slot.empty:
                continue
            # Random
            rc = slot.sample(1, random_state=random.randint(0, 99999)).iloc[0]
            r_total += 1
            if rc['RSSI'] > -900:
                r_ok += 1
                r_rssi.append(rc['RSSI'])
                r_snr.append(rc['SNR'])
            # TinyML (best channel)
            valid = slot[slot['RSSI'] > -900]
            m_total += 1
            if not valid.empty:
                best = valid.loc[valid['RSSI'].idxmax()]
                m_ok += 1
                m_rssi.append(best['RSSI'])
                m_snr.append(best['SNR'])

        results.append({
            'Payload_Size': size,
            'RSSI_Random': round(float(np.mean(r_rssi)), 1) if r_rssi else -120.0,
            'RSSI_TinyML': round(float(np.mean(m_rssi)), 1) if m_rssi else -120.0,
            'SNR_Random': round(float(np.mean(r_snr)), 1) if r_snr else -20.0,
            'SNR_TinyML': round(float(np.mean(m_snr)), 1) if m_snr else -20.0,
            'PDR_Fixed': round(pdr_fixed, 3),
            'PDR_Random': round(r_ok / r_total, 3) if r_total > 0 else 0.0,
            'PDR_TinyML': round(m_ok / m_total, 3) if m_total > 0 else 0.0,
        })

    res_df = pd.DataFrame(results)
    print(res_df.to_string(index=False))

    # Vẽ biểu đồ
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    x = np.arange(len(payload_sizes))
    w = 0.35

    axes[0].bar(x - w/2, res_df['RSSI_Random'], w, label='Random', color='#1f77b4')
    axes[0].bar(x + w/2, res_df['RSSI_TinyML'], w, label='TinyML', color='#2ca02c')
    axes[0].set_title('(a) Average RSSI (dBm)', fontsize=12)
    axes[0].set_xlabel('Payload Size (Bytes)')
    axes[0].set_ylabel('RSSI (dBm)')
    axes[0].set_xticks(x); axes[0].set_xticklabels(payload_sizes)
    axes[0].legend(); axes[0].grid(axis='y', linestyle='--', alpha=0.7)

    axes[1].bar(x - w/2, res_df['SNR_Random'], w, label='Random', color='#1f77b4')
    axes[1].bar(x + w/2, res_df['SNR_TinyML'], w, label='TinyML', color='#2ca02c')
    axes[1].set_title('(b) Average SNR (dB)', fontsize=12)
    axes[1].set_xlabel('Payload Size (Bytes)')
    axes[1].set_ylabel('SNR (dB)')
    axes[1].set_xticks(x); axes[1].set_xticklabels(payload_sizes)
    axes[1].legend(); axes[1].grid(axis='y', linestyle='--', alpha=0.7)

    axes[2].plot(x, res_df['PDR_Fixed'], 'r--o', label='Fixed', linewidth=2)
    axes[2].plot(x, res_df['PDR_Random'], 'b--s', label='Random', linewidth=2)
    axes[2].plot(x, res_df['PDR_TinyML'], 'g-^', label='TinyML', linewidth=2.5)
    axes[2].set_title('(c) Packet Delivery Ratio', fontsize=12)
    axes[2].set_xlabel('Payload Size (Bytes)')
    axes[2].set_ylabel('PDR')
    axes[2].set_ylim([0.0, 1.05])
    axes[2].set_xticks(x); axes[2].set_xticklabels(payload_sizes)
    axes[2].legend(); axes[2].grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, "comparison_metrics.png"), dpi=300)
    plt.show()

    res_df.to_csv(os.path.join(FIGURES_DIR, "comparison_results.csv"), index=False)
    print("[SUCCESS] Đã lưu biểu đồ và dữ liệu.")


## 8. Bước 6: Benchmark kiến trúc mạng nơ-ron (`06_benchmark_models`)
So sánh 5 kiến trúc: FCNN, 1D-CNN, GRU, Separable CNN, ResNet-1D.

> **Lưu ý:** Bước này mất nhiều thời gian. Có thể chạy riêng bằng `python scripts/06_benchmark_models.py`.


In [ ]:
# Nạp dữ liệu từ MLdataready (433 MHz Channels)
CH_DATA = ["433_175", "433_375", "433_575"]
ml_headers = ['packet_ID', 'CH1', 'CH2', 'CH3', 'size_of_message', 'RSSI', 'SNR', 'frequency']
SAMPLES = 10
NUM_CH = 3
ONE_HOT_CH = np.eye(NUM_CH)

raw_in, raw_out = [], []
for ci in range(NUM_CH):
    ch = CH_DATA[ci]
    p = os.path.join(MLDATAREADY_DIR, "ABC", f"{ch}.csv")
    if not os.path.exists(p):
        p = os.path.join(MLDATAREADY_DIR, "A", f"{ch}.csv")
    if not os.path.exists(p):
        continue
    df_ch = pd.read_csv(p, names=ml_headers, skiprows=1)
    n_rec = int(df_ch.shape[0] / SAMPLES)
    for i in range(n_rec):
        t = []
        for j in range(SAMPLES):
            idx = i * SAMPLES + j
            t.append([float(df_ch[c][idx]) for c in ['CH1','CH2','CH3','SNR','RSSI','size_of_message','packet_ID','frequency']])
        raw_in.append(t)
        raw_out.append(ONE_HOT_CH[ci])

raw_in = np.array(raw_in)
raw_out = np.array(raw_out)
N_b, T_b, F_b = raw_in.shape
print(f"[DATA] Benchmark: {N_b} mẫu, {T_b}x{F_b}")

# Chuẩn hóa
flat_b = raw_in.reshape(N_b * T_b, F_b)
scaler_b = StandardScaler()
X_seq_b = scaler_b.fit_transform(flat_b).reshape(N_b, T_b, F_b)
X_flat_b = X_seq_b.reshape(N_b, T_b * F_b)

idx_tr, idx_tmp = train_test_split(np.arange(N_b), test_size=0.4, random_state=SEED)
idx_vl, idx_ts = train_test_split(idx_tmp, test_size=0.5, random_state=SEED)
y_tr_b, y_vl_b, y_ts_b = raw_out[idx_tr], raw_out[idx_vl], raw_out[idx_ts]

# Định nghĩa các mô hình (rút gọn)
def mk_fcnn():
    return tf.keras.Sequential([layers.Input(shape=(T_b*F_b,)), layers.Dense(10, activation='relu'),
        layers.Dense(10, activation='relu', kernel_regularizer=regularizers.L1(1e-4)),
        layers.Dense(NUM_CH, activation='softmax')], name="Baseline_FCNN"), True

def mk_cnn1d():
    return tf.keras.Sequential([layers.Input(shape=(T_b, F_b)),
        layers.Conv1D(16, 3, padding='same', activation='relu'), layers.BatchNormalization(),
        layers.MaxPooling1D(2), layers.Conv1D(32, 3, padding='same', activation='relu'),
        layers.GlobalAveragePooling1D(), layers.Dense(16, activation='relu'),
        layers.Dense(NUM_CH, activation='softmax')], name="1D_CNN"), False

def mk_sep():
    return tf.keras.Sequential([layers.Input(shape=(T_b, F_b)),
        layers.SeparableConv1D(16, 3, padding='same', activation='relu'),
        layers.SeparableConv1D(32, 3, padding='same', activation='relu'),
        layers.GlobalAveragePooling1D(),
        layers.Dense(NUM_CH, activation='softmax')], name="Separable_1D_CNN"), False

bench_results = []
for factory in [mk_fcnn, mk_cnn1d, mk_sep]:
    mdl, is_flat = factory()
    mdl.compile(optimizer=tf.keras.optimizers.Adam(0.003), loss='categorical_crossentropy', metrics=['accuracy'])
    xtr = X_flat_b[idx_tr] if is_flat else X_seq_b[idx_tr]
    xvl = X_flat_b[idx_vl] if is_flat else X_seq_b[idx_vl]
    xts = X_flat_b[idx_ts] if is_flat else X_seq_b[idx_ts]
    mdl.fit(xtr, y_tr_b, epochs=100, batch_size=8, validation_data=(xvl, y_vl_b), verbose=0)
    loss_b, acc_b = mdl.evaluate(xts, y_ts_b, verbose=0)
    bench_results.append({'Model': mdl.name, 'Accuracy': f"{acc_b*100:.1f}%", 'Params': mdl.count_params()})
    print(f"  {mdl.name:<20} | Acc: {acc_b*100:.1f}% | Params: {mdl.count_params()}")

pd.DataFrame(bench_results)


## 9. Kết luận

Pipeline TinyML hoàn chỉnh đã được thực hiện:
- **Thu thập** → **Tiền xử lý** → **Huấn luyện** → **Triển khai** → **Đánh giá**
- Mô hình FCNN Baseline đạt kích thước ~5.8 KB TFLite, phù hợp Flash ESP32
- TinyML Channel Hopping vượt trội Random Hopping về PDR, RSSI, SNR
- Separable 1D-CNN là ứng viên tiềm năng cho MCU hiệu năng cao hơn

### Đường dẫn artifacts:
- Models: `models/channel_model.tflite`, `models/model.h`
- Figures: `figures/training_trends.png`, `figures/comparison_metrics.png`
